# ExpW 4-Class — Improved Training (EfficientNet-B0, PyTorch, CUDA, AMP)
This notebook has the the improvements:
- Crop-to-disk cache from `label.lst` + `origin` images (fast re-runs)
- Stronger augmentations (RandomResizedCrop, ColorJitter, RandomGrayscale, RandomErasing)
- Label smoothing
- Two-phase training: freeze backbone → unfreeze fine-tune (lower LR)
- Early stopping on validation accuracy + best-checkpoint saving
- Class imbalance handling ( WeightedRandomSampler + optional class-weighted loss)
- Full evaluation: Accuracy, Precision, Recall, F1 + Confusion Matrix
- Real-time webcam inference 

##  paths 
- Images: `D:\expWdataset\origin\origin`
- Labels: `D:\expWdataset\label.lst`


In [3]:


import os, random
from pathlib import Path
from typing import Dict
import numpy as np
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import cv2
except Exception:
    cv2 = None
    print("OpenCV not available. Real-time webcam cell will require: pip install opencv-python")

# Helps with truncated/corrupt JPEGs (common in big web datasets)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [5]:
# -------------------------
# CONFIG
# -------------------------
IMG_ROOT = r"D:\expWdataset\origin\origin"
LABEL_LST = r"D:\expWdataset\label.lst"

# Cropped-face cache directory (created if not exists)
CACHE_ROOT = r"D:\expWdataset\expw_4class_crops"

# Keep only 4 emotions for MER: anger, happy, sad, neutral
KEEP: Dict[int, str] = {
    0: "anger",
    3: "happy",
    4: "sad",
    6: "neutral",
}

# Filter low-confidence face boxes (set None to disable)
MIN_CONF = 0.80

# Split ratios (stable split via hashing image+face_id)
TRAIN_P = 0.80
VAL_P   = 0.10
TEST_P  = 0.10

# Crop size saved to disk
CROP_SAVE_SIZE = 224

# Training params
BATCH_SIZE = 64
NUM_WORKERS = 0  # notebook-safe (Windows/Jupyter). Use >0 in .py scripts if stable.
PIN_MEMORY = True if device.type == "cuda" else False

# Phase 1: train head only
EPOCHS_FROZEN = 3
LR_FROZEN = 3e-4

# Phase 2: fine-tune all layers
EPOCHS_FINETUNE = 10
LR_FINETUNE = 1e-4

WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# Early stopping
PATIENCE = 3  # stop if val acc doesn't improve for N epochs

# Imbalance handling
USE_WEIGHTED_SAMPLER = True      # good default for ExpW
USE_CLASS_WEIGHTED_LOSS = False  # optional alternative to sampler

# AMP (mixed precision) for faster GPU training
USE_AMP = True if device.type == "cuda" else False

# Real-time webcam
WEBCAM_INDEX = 0
REALTIME_INPUT_SIZE = 224


## Step 1 — Build cropped-face cache 
Re-runs will skip the already-cached crops.


In [7]:
import hashlib

def safe_crop(img: Image.Image, left: int, top: int, right: int, bottom: int):
    w, h = img.size
    left = max(0, min(left, w - 1))
    right = max(1, min(right, w))
    top = max(0, min(top, h - 1))
    bottom = max(1, min(bottom, h))
    if right <= left or bottom <= top:
        # fallback: center square crop
        side = min(w, h)
        left0 = (w - side) // 2
        top0  = (h - side) // 2
        return img.crop((left0, top0, left0 + side, top0 + side))
    return img.crop((left, top, right, bottom))

def split_for_key(key: str) -> str:
    # Stable split by hashing image+face_id so re-running produces identical splits
    h = hashlib.md5(key.encode("utf-8")).hexdigest()
    r = int(h[:8], 16) / 0xFFFFFFFF
    if r < TRAIN_P:
        return "train"
    if r < TRAIN_P + VAL_P:
        return "val"
    return "test"

def build_crop_cache():
    img_root = Path(IMG_ROOT)
    label_path = Path(LABEL_LST)
    cache_root = Path(CACHE_ROOT)

    if not img_root.exists():
        raise FileNotFoundError(f"IMG_ROOT not found: {img_root}")
    if not label_path.exists():
        raise FileNotFoundError(f"LABEL_LST not found: {label_path}")

    for split in ["train", "val", "test"]:
        for cls in KEEP.values():
            (cache_root / split / cls).mkdir(parents=True, exist_ok=True)

    total = kept = missing = bad = 0
    with open(label_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    pbar = tqdm(lines, desc="Cropping faces to cache")
    for line in pbar:
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 8:
            bad += 1
            continue

        total += 1
        image_name = parts[0]
        face_id = parts[1]

        try:
            top = int(float(parts[2]))
            left = int(float(parts[3]))
            right = int(float(parts[4]))
            bottom = int(float(parts[5]))
            conf = float(parts[6])
            label = int(parts[7])
        except Exception:
            bad += 1
            continue

        if label not in KEEP:
            continue
        if MIN_CONF is not None and conf < MIN_CONF:
            continue

        img_path = img_root / image_name
        if not img_path.exists():
            missing += 1
            continue

        key = f"{image_name}|{face_id}"
        split = split_for_key(key)
        cls = KEEP[label]

        out_name = f"{Path(image_name).stem}_face{face_id}.jpg"
        out_path = cache_root / split / cls / out_name

        if out_path.exists():
            kept += 1
            continue

        try:
            img = Image.open(img_path).convert("RGB")
            crop = safe_crop(img, left, top, right, bottom)
            crop = crop.resize((CROP_SAVE_SIZE, CROP_SAVE_SIZE), Image.BILINEAR)
            crop.save(out_path, quality=95)
            kept += 1
        except Exception:
            bad += 1
            continue

        if total % 5000 == 0:
            pbar.set_postfix({"saved": kept, "missing": missing, "bad": bad})

    print("Cache build complete.")
    print(f"Rows read: {total}")
    print(f"Saved/available crops (4-class): {kept}")
    print(f"Missing images: {missing}")
    print(f"Bad/failed rows: {bad}")
    print("Cache root:", CACHE_ROOT)

# Run ONCE (re-runs will skip already-cached crops)
build_crop_cache()


Cropping faces to cache:   0%|          | 0/91793 [00:00<?, ?it/s]

Cache build complete.
Rows read: 91793
Saved/available crops (4-class): 79538
Missing images: 0
Bad/failed rows: 0
Cache root: D:\expWdataset\expw_4class_crops


## Step 2 — Dataloaders + imbalance handling


In [9]:
# -------------------------
# DATA LOADERS (from cached crops) + imbalance handling
# -------------------------
cache_root = Path(CACHE_ROOT)
train_dir = cache_root / "train"
val_dir   = cache_root / "val"
test_dir  = cache_root / "test"

for d in [train_dir, val_dir, test_dir]:
    if not d.exists():
        raise FileNotFoundError(f"Missing cached split folder: {d}")

weights = EfficientNet_B0_Weights.DEFAULT
imagenet_mean = weights.transforms().mean
imagenet_std  = weights.transforms().std

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.RandomGrayscale(p=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3)),
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_ds = ImageFolder(train_dir, transform=train_tfms)
val_ds   = ImageFolder(val_dir,   transform=eval_tfms)
test_ds  = ImageFolder(test_dir,  transform=eval_tfms)

print("Classes:", train_ds.classes)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))
print("Test samples:", len(test_ds))

# Class distribution
counts = np.zeros(len(train_ds.classes), dtype=np.int64)
for _, y in train_ds.samples:
    counts[y] += 1

print("\nTrain class counts:")
for i, c in enumerate(train_ds.classes):
    print(f"{c:8s}: {counts[i]}")
print("Imbalance ratio (max/min):", counts.max() / max(1, counts.min()))

sampler = None
shuffle = True
class_weights = None

if USE_WEIGHTED_SAMPLER:
    w_per_class = 1.0 / np.maximum(counts, 1)
    sample_weights = np.array([w_per_class[y] for _, y in train_ds.samples], dtype=np.float32)
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    shuffle = False
    print("\nUsing WeightedRandomSampler.")

if USE_CLASS_WEIGHTED_LOSS:
    class_weights = torch.tensor((counts.sum() / np.maximum(counts, 1)), dtype=torch.float32)
    class_weights = class_weights / class_weights.sum() * len(counts)
    print("\nClass-weighted loss weights:", class_weights.tolist())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


Classes: ['anger', 'angry', 'happy', 'neutral', 'sad']
Train samples: 122907
Val samples: 15259
Test samples: 15349

Train class counts:
anger   : 2918
angry   : 2847
happy   : 46351
neutral : 54242
sad     : 16549
Imbalance ratio (max/min): 19.052335792061818

Using WeightedRandomSampler.


## Step 3 — Model + label smoothing + AMP


In [11]:
# -------------------------
# MODEL + LOSS (label smoothing) + AMP
# -------------------------
num_classes = len(train_ds.classes)

model = efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

if USE_CLASS_WEIGHTED_LOSS and class_weights is not None:
    class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def set_backbone_trainable(trainable: bool):
    for name, p in model.named_parameters():
        if "classifier" in name:
            p.requires_grad = True
        else:
            p.requires_grad = trainable

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


C:\Users\Jugantar\AppData\Local\Temp\ipykernel_16976\2374437013.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## Step 4 — Two-phase training + early stopping


In [19]:
# -------------------------
# TRAINING (two phases + early stopping + best checkpoint)
# -------------------------
from time import time

best_val_acc = 0.0
patience_left = PATIENCE
best_path = Path(CACHE_ROOT) / "best_efficientnet_b0_4class_improved.pt"

def run_epoch(loader, train_mode: bool, optimizer=None):
    if train_mode:
        model.train()
    else:
        model.eval()

    epoch_loss = 0.0
    correct = 0
    n = 0

    pbar = tqdm(loader, desc=("Train" if train_mode else "Val"))
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

        bs = x.size(0)
        epoch_loss += loss.item() * bs
        correct += (torch.argmax(logits, dim=1) == y).sum().item()
        n += bs
        pbar.set_postfix({"loss": epoch_loss / max(1,n), "acc": correct / max(1,n)})

    return epoch_loss / n, correct / n

def fit(num_epochs: int, lr: float, train_backbone: bool):
    global best_val_acc, patience_left

    set_backbone_trainable(train_backbone)
    params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    for epoch in range(1, num_epochs + 1):
        t0 = time()
        train_loss, train_acc = run_epoch(train_loader, train_mode=True, optimizer=optimizer)
        val_loss, val_acc = run_epoch(val_loader, train_mode=False)
        scheduler.step()
        dt = time() - t0

        print(f"Epoch {epoch} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.4f} | time {dt:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_left = PATIENCE
            torch.save({
                "model_state": model.state_dict(),
                "classes": train_ds.classes,
                "class_to_idx": train_ds.class_to_idx,
                "mean": list(imagenet_mean),
                "std": list(imagenet_std),
                "best_val_acc": best_val_acc,
                "min_conf": MIN_CONF,
                "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
                "use_class_weighted_loss": USE_CLASS_WEIGHTED_LOSS,
            }, best_path)
            print("Saved best checkpoint:", best_path, "| best val acc:", best_val_acc)
        else:
            patience_left -= 1
            print("No improvement. Patience left:", patience_left)
            if patience_left <= 0:
                print("Early stopping triggered.")
                return

print("Phase 1: train classifier head only (backbone frozen)")
fit(EPOCHS_FROZEN, lr=LR_FROZEN, train_backbone=False)

print("\nPhase 2: fine-tune full model (backbone unfrozen)")
fit(EPOCHS_FINETUNE, lr=LR_FINETUNE, train_backbone=True)

print("\nDone. Best val acc:", best_val_acc)
print("Best checkpoint:", best_path)


Phase 1: train classifier head only (backbone frozen)


Train:   0%|          | 0/1921 [00:00<?, ?it/s]

C:\Users\Jugantar\AppData\Local\Temp\ipykernel_16976\2946570239.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


RuntimeError: Input type (torch.cuda.HalfTensor) and weight type (torch.FloatTensor) should be the same

## Step 5 — Evaluation (metrics + confusion matrix)


In [13]:
# -------------------------
# EVALUATION (TEST): metrics + confusion matrix
# -------------------------
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

classes = ckpt["classes"]
print("Loaded classes:", classes)
print("Best val acc (ckpt):", ckpt.get("best_val_acc"))

all_preds, all_true = [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test"):
        x = x.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_true.extend(y.numpy().tolist())

acc = accuracy_score(all_true, all_preds)
print("\nTest Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=classes, digits=4))

cm = confusion_matrix(all_true, all_preds)
print("\nConfusion Matrix:\n", cm)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(ticks=np.arange(len(classes)), labels=classes, rotation=45, ha="right")
plt.yticks(ticks=np.arange(len(classes)), labels=classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()


NameError: name 'best_path' is not defined

## Step 6 — Real-time webcam inference (optional)


In [17]:
# -------------------------
# REAL-TIME WEBCAM INFERENCE (press 'q' to quit)
# -------------------------
if cv2 is None:
    raise RuntimeError("OpenCV not installed. Run: pip install opencv-python")

mean = np.array(ckpt["mean"], dtype=np.float32)
std  = np.array(ckpt["std"], dtype=np.float32)

def preprocess_frame(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb).resize((REALTIME_INPUT_SIZE, REALTIME_INPUT_SIZE))
    x = np.asarray(img).astype(np.float32) / 255.0
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1))
    return torch.from_numpy(x).unsqueeze(0)

cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Try WEBCAM_INDEX=1 or 2.")

model.eval()
with torch.no_grad():
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        x = preprocess_frame(frame).to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        pred = int(np.argmax(probs))
        label = classes[pred]
        conf = float(probs[pred])
        cv2.putText(frame, f"{label} ({conf:.2f})", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
        cv2.imshow("ExpW 4-class - Improved EfficientNetB0", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


C:\Users\Jugantar\AppData\Local\Temp\ipykernel_32984\2916231984.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


## Step 7 — Export TorchScript 


In [15]:
import torch

model.eval()
model.cpu()

dummy = torch.randn(1, 3, 224, 224)  # input size same as training

torch.onnx.export(
    model,
    dummy,
    "fer_expw_efficientnet_b0_4class.onnx",
    input_names=["input"],
    output_names=["logits"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
)

print("Saved ONNX: fer_expw_efficientnet_b0_4class.onnx")

OnnxExporterError: Module onnx is not installed!

In [19]:
# Export TorchScript (Pi-safe CPU trace)
from pathlib import Path
import torch

export_path = Path(CACHE_ROOT) / "efficientnet_pi_safe.pt"

model.eval()
model = model.cpu()  # IMPORTANT: force CPU weights

example = torch.randn(1, 3, 224, 224, device="cpu")  # IMPORTANT: CPU input

with torch.no_grad():
    traced = torch.jit.trace(model, example, strict=False)

traced = torch.jit.freeze(traced)  # optional but helps stability/perf
traced.save(str(export_path))

print("Saved Pi-safe TorchScript to:", export_path)

Saved Pi-safe TorchScript to: D:\expWdataset\expw_4class_crops\efficientnet_pi_safe.pt


In [9]:
# -------------------------
#  Export TorchScript (deployment)
# -------------------------
export_path = Path(CACHE_ROOT) / "efficientnet_b0_4class_improved_torchscript.pt"
model.eval()
example = torch.randn(1, 3, 224, 224).to(device)
traced = torch.jit.trace(model, example)
traced.save(str(export_path))
print("Saved TorchScript to:", export_path)


Saved TorchScript to: D:\expWdataset\expw_4class_crops\efficientnet_b0_4class_improved_torchscript.pt
